In [1]:
!pip install pyspark==3.5.3 apache-sedona==1.7.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 66.0 MB/s  0:00:04
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840690 sha256=a0da965a827c2c52dae27c40a842e5a8df7ef22847b3e4cb1f34b457b4574fc0
  Stored in directory: /root/.cache/pip/wheels/57/01/5d/dd344b64ae939cd8ea67ffa0abd6aed72f14b1165228e281a0
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.5
    Uninstalling pyspark-4.0.5:
      Successfully uninstalled pyspark-4.0.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [apache-sedona]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behavi

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
import os

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from sedona.spark import SedonaContext

spark = (
    SparkSession.builder
    .appName("GeoPulse-Sedona")
    .config("spark.jars.packages",
            "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.7.1,"
            "org.datasyslab:geotools-wrapper:1.7.1-28.5")
    .getOrCreate()
)

sedona = SedonaContext.create(spark)

print("Spark version:", spark.version)
print("Apache Sedona initialized successfully!")

Spark version: 3.5.3
Apache Sedona initialized successfully!


In [4]:
from pathlib import Path

repo_path = Path("/content/gdrive/MyDrive/GeoPulse-GitHub-Repo")

gps_file = repo_path / "data/synthetic/synthetic_gps_data.csv"

gps_df = spark.read.csv(
    str(gps_file),
    header=True,
    inferSchema=True
)

print("GPS dataset loaded successfully!")
print("Total records:", gps_df.count())

gps_df.show(5, truncate=False)

GPS dataset loaded successfully!
Total records: 50000
+------------+------------------+-----------------+-------------------+
|DeviceID    |latitude          |longitude        |timestamp          |
+------------+------------------+-----------------+-------------------+
|DEVICE_00001|18.552963223680592|73.78487346516302|2026-11-17 01:32:09|
|DEVICE_00001|18.64239615694114 |73.94378320584028|2026-11-17 04:22:55|
|DEVICE_00001|18.4013805292809  |73.95386535711371|2026-11-17 10:18:30|
|DEVICE_00001|18.593061192324164|73.79967892038354|2026-11-17 12:12:06|
|DEVICE_00001|18.49716932242237 |73.81783725794348|2026-11-17 13:35:23|
+------------+------------------+-----------------+-------------------+
only showing top 5 rows



In [6]:
import os

print("Repository exists:", repo_path.exists())
print("Repository contents:")
print(os.listdir(repo_path))

print("\nSynthetic folder contents:")
synthetic_path = repo_path / "data" / "synthetic"

if synthetic_path.exists():
    print(os.listdir(synthetic_path))
else:
    print("data/synthetic folder not found")

Repository exists: True
Repository contents:
['notebooks', 'scripts', 'data_generator', 'tests', 'data', 'dbt_project', 'airflow', 'frontend', 'snowflake', 'analytics', 'spatial_engine', 'docs', '.git', 'README.md']

Synthetic folder contents:
['synthetic_gps_data.csv', 'realistic_urban_mobility.csv']


In [9]:
import os

notebooks_path = repo_path / "notebooks"

print(os.listdir(notebooks_path))

['06_apache_sedona_pyspark_integration.ipynb', '01_project_setup.ipynb', '02_synthetic_gps_data_generator.ipynb', '03_realistic_urban_mobility_patterns.ipynb', '04_retail_locations_and_zoning.ipynb', '05_snowflake_geospatial_warehouse.ipynb']


In [11]:
import pandas as pd
import numpy as np
import random

store_categories = [
    "Supermarket",
    "Fashion",
    "Electronics",
    "Convenience",
    "Lifestyle"
]

stores = []

for i in range(1, 21):
    stores.append({
        "StoreID": f"GP_STORE_{i:03d}",
        "StoreName": f"GeoPulse Retail Store {i}",
        "latitude": np.random.uniform(18.40, 18.65),
        "longitude": np.random.uniform(73.75, 74.00),
        "Category": random.choice(store_categories),
        "StoreSize_sqft": random.randint(800, 8000)
    })

store_df = pd.DataFrame(stores)

store_df.to_csv(synthetic_path / "store_locations.csv", index=False)

print("Synthetic store dataset recreated and saved!")
print("Total stores:", len(store_df))
display(store_df.head())

Synthetic store dataset recreated and saved!
Total stores: 20


,StoreID,StoreName,latitude,longitude,Category,StoreSize_sqft
0,GP_STORE_001,GeoPulse Retail Store 1,18.451074,73.830168,Fashion,5312
1,GP_STORE_002,GeoPulse Retail Store 2,18.566731,73.873833,Convenience,3710
2,GP_STORE_003,GeoPulse Retail Store 3,18.536793,73.869998,Fashion,6319
3,GP_STORE_004,GeoPulse Retail Store 4,18.515947,73.913068,Electronics,965
4,GP_STORE_005,GeoPulse Retail Store 5,18.423869,73.819911,Electronics,2437


In [12]:
store_file = repo_path / "data" / "synthetic" / "store_locations.csv"

stores_df = spark.read.csv(
    str(store_file),
    header=True,
    inferSchema=True
)

print("Store dataset loaded successfully!")
print("Total stores:", stores_df.count())

stores_df.show(5, truncate=False)

Store dataset loaded successfully!
Total stores: 20
+------------+-----------------------+------------------+-----------------+-----------+--------------+
|StoreID     |StoreName              |latitude          |longitude        |Category   |StoreSize_sqft|
+------------+-----------------------+------------------+-----------------+-----------+--------------+
|GP_STORE_001|GeoPulse Retail Store 1|18.451073698491776|73.83016804566476|Fashion    |5312          |
|GP_STORE_002|GeoPulse Retail Store 2|18.566731396907343|73.87383264486247|Convenience|3710          |
|GP_STORE_003|GeoPulse Retail Store 3|18.536793266273623|73.86999768645072|Fashion    |6319          |
|GP_STORE_004|GeoPulse Retail Store 4|18.515947091062827|73.9130675653856 |Electronics|965           |
|GP_STORE_005|GeoPulse Retail Store 5|18.423869171637296|73.81991114173908|Electronics|2437          |
+------------+-----------------------+------------------+-----------------+-----------+--------------+
only showing top 5 ro

In [14]:
gps_df.createOrReplaceTempView("gps_data")
stores_df.createOrReplaceTempView("store_data")

gps_spatial_df = spark.sql("""
    SELECT *,
           ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE)) AS geometry
    FROM gps_data
""")

stores_spatial_df = spark.sql("""
    SELECT *,
           ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE)) AS geometry
    FROM store_data
""")

print("Spatial geometries created successfully!")

gps_spatial_df.select("DeviceID", "latitude", "longitude", "geometry").show(5, truncate=False)

stores_spatial_df.select("StoreID", "latitude", "longitude", "geometry").show(5, truncate=False)

Spatial geometries created successfully!
+------------+------------------+-----------------+--------------------------------------------+
|DeviceID    |latitude          |longitude        |geometry                                    |
+------------+------------------+-----------------+--------------------------------------------+
|DEVICE_00001|18.552963223680592|73.78487346516302|POINT (73.78487346516302 18.552963223680592)|
|DEVICE_00001|18.64239615694114 |73.94378320584028|POINT (73.94378320584028 18.64239615694114) |
|DEVICE_00001|18.4013805292809  |73.95386535711371|POINT (73.95386535711371 18.4013805292809)  |
|DEVICE_00001|18.593061192324164|73.79967892038354|POINT (73.79967892038354 18.593061192324164)|
|DEVICE_00001|18.49716932242237 |73.81783725794348|POINT (73.81783725794348 18.49716932242237) |
+------------+------------------+-----------------+--------------------------------------------+
only showing top 5 rows

+------------+------------------+-----------------+----------

In [15]:
print("GPS spatial DataFrame schema:")
gps_spatial_df.printSchema()

print("\nStore spatial DataFrame schema:")
stores_spatial_df.printSchema()

print("\nGPS spatial records:", gps_spatial_df.count())
print("Store spatial records:", stores_spatial_df.count())

GPS spatial DataFrame schema:
root
 |-- DeviceID: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- geometry: geometry (nullable = true)


Store spatial DataFrame schema:
root
 |-- StoreID: string (nullable = true)
 |-- StoreName: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- Category: string (nullable = true)
 |-- StoreSize_sqft: integer (nullable = true)
 |-- geometry: geometry (nullable = true)


GPS spatial records: 50000
Store spatial records: 20


## Day 6 – PySpark and Apache Sedona Integration

### Objective
Configure Apache Spark and Apache Sedona for distributed geospatial processing and load synthetic GPS and retail location datasets.

### Tasks Completed
- Installed PySpark 3.5.3 and Apache Sedona 1.7.1.
- Configured and initialized the Spark session with Sedona.
- Loaded 50,000 synthetic GPS records into a Spark DataFrame.
- Recreated and loaded 20 synthetic retail store records.
- Created spatial point geometries for GPS ping coordinates and store coordinates using Sedona SQL.
- Validated spatial DataFrame schemas and record counts.

### Validation Results
| Dataset | Record Count | Status |
|---|---:|---|
| GPS Data | 50,000 | Validated |
| Store Locations | 20 | Validated |
| GPS Geometry | 50,000 | Created |
| Store Geometry | 20 | Created |

### Tools & Technologies
PySpark 3.5.3, Apache Sedona 1.7.1, Spark SQL, Google Colab, Google Drive, Python.

### Outcome
Successfully configured a geospatial processing environment and created spatial DataFrames for synthetic mobility and retail location data, preparing the foundation for spatial joins and catchment analysis.

*All datasets used in this project are synthetic and are intended for analytical demonstration purposes.*


In [18]:
from pathlib import Path
import subprocess

repo_path = Path("/content/gdrive/MyDrive/GeoPulse-GitHub-Repo")
day6_path = repo_path / "notebooks" / "06_apache_sedona_pyspark_integration.ipynb"

print("Notebook exists:", day6_path.exists())

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print("\nGit status:")
print(result.stdout)

Notebook exists: True

Git status:
?? data/synthetic/store_locations.csv
?? notebooks/04_retail_locations_and_zoning.ipynb
?? notebooks/05_snowflake_geospatial_warehouse.ipynb
?? notebooks/06_apache_sedona_pyspark_integration.ipynb



In [19]:
files_to_stage = [
    "notebooks/04_retail_locations_and_zoning.ipynb",
    "notebooks/05_snowflake_geospatial_warehouse.ipynb",
    "notebooks/06_apache_sedona_pyspark_integration.ipynb",
    "data/synthetic/store_locations.csv"
]

result = subprocess.run(
    ["git", "add"] + files_to_stage,
    cwd=repo_path,
    capture_output=True,
    text=True
)

print("Files staged!" if result.returncode == 0 else result.stderr)

subprocess.run(["git", "status", "--short"], cwd=repo_path)

Files staged!


CompletedProcess(args=['git', 'status', '--short'], returncode=0)

In [20]:
import subprocess

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=repo_path,
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else "No pending changes.")

A  data/synthetic/store_locations.csv
A  notebooks/04_retail_locations_and_zoning.ipynb
A  notebooks/05_snowflake_geospatial_warehouse.ipynb
AM notebooks/06_apache_sedona_pyspark_integration.ipynb

